In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ── SQL SETUP: Load data into SQLite ──

import pandas as pd
import sqlite3

# Load both datasets
customer_df = pd.read_csv('/content/drive/MyDrive/retail-churn-project/data/customer_churn_features.csv')
transactions_df = pd.read_csv('/content/drive/MyDrive/retail-churn-project/data/online_retail_II.csv')

# Create SQLite database
conn = sqlite3.connect('retail_churn.db')

# Save both tables into the database
customer_df.to_sql('customers', conn, if_exists='replace', index=False)
transactions_df.to_sql('transactions', conn, if_exists='replace', index=False)

print("✅ Database created with tables:")
print("   → customers:", len(customer_df), "rows")
print("   → transactions:", len(transactions_df), "rows")

✅ Database created with tables:
   → customers: 5878 rows
   → transactions: 1067371 rows


In [ ]:
# Helper function to run queries neatly
def run_query(title, query):
    print(f"\n{'='*50}")
    print(f"📊 {title}")
    print('='*50)
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))
    return result

# ── QUERY 1: Churn rate by country (top 10) ──
run_query("Churn Rate by Country (Top 10)", """
    SELECT t.Country,
           COUNT(DISTINCT t.[Customer ID]) AS TotalCustomers,
           SUM(c.Churned) AS ChurnedCustomers,
           ROUND(SUM(c.Churned) * 100.0 / COUNT(DISTINCT t.[Customer ID]), 2) AS ChurnRate_pct
    FROM transactions t
    JOIN customers c ON t.[Customer ID] = c.[Customer ID]
    GROUP BY t.Country
    HAVING TotalCustomers > 10
    ORDER BY ChurnRate_pct DESC
    LIMIT 10
""")

# ── QUERY 2: Average order value by risk segment ──
run_query("Avg Order Value by Risk Segment", """
    SELECT RiskSegment,
           COUNT(*) AS CustomerCount,
           ROUND(AVG(AvgOrderValue), 2) AS AvgOrderValue,
           ROUND(AVG(Monetary), 2) AS AvgTotalSpend
    FROM customers
    GROUP BY RiskSegment
    ORDER BY AvgOrderValue DESC
""")

# ── QUERY 3: Top 10 products bought by loyal customers ──
run_query("Top 10 Products by Loyal Customers", """
    SELECT t.Description,
           COUNT(*) AS TimesPurchased,
           ROUND(SUM(t.Quantity * t.Price), 2) AS TotalRevenue
    FROM transactions t
    JOIN customers c ON t.[Customer ID] = c.[Customer ID]
    WHERE c.Churned = 0
    GROUP BY t.Description
    ORDER BY TimesPurchased DESC
    LIMIT 10
""")

# ── QUERY 4: Monthly revenue trend ──
run_query("Monthly Revenue Trend", """
    SELECT SUBSTR(InvoiceDate, 1, 7) AS Month,
           ROUND(SUM(Quantity * Price), 2) AS TotalRevenue,
           COUNT(DISTINCT [Customer ID]) AS ActiveCustomers
    FROM transactions
    GROUP BY Month
    ORDER BY Month
""")

# ── QUERY 5: High value churned customers (recovery targets) ──
run_query("High Value Churned Customers - Recovery Targets", """
    SELECT [Customer ID],
           ROUND(Monetary, 2) AS TotalSpend,
           Frequency AS TotalOrders,
           ROUND(ChurnProbability * 100, 1) AS ChurnProb_pct
    FROM customers
    WHERE Churned = 1 AND Monetary > 5000
    ORDER BY Monetary DESC
    LIMIT 10
""")


📊 Churn Rate by Country (Top 10)
        Country  TotalCustomers  ChurnedCustomers  ChurnRate_pct
Channel Islands              13               808        6215.38
    Switzerland              22               935        4250.00
         Sweden              19               701        3689.47
 United Kingdom            5350            169904        3175.78
    Netherlands              22               697        3168.18
       Portugal              24               735        3062.50
          Italy              17               474        2788.24
        Austria              13               300        2307.69
        Germany             107              2382        2226.17
      Australia              15               285        1900.00

📊 Avg Order Value by Risk Segment
RiskSegment  CustomerCount  AvgOrderValue  AvgTotalSpend
  High Risk           2606          52.22         931.82
   Low Risk           2627          48.60        5009.37
Medium Risk            582          31.86    

,Customer ID,TotalSpend,TotalOrders,ChurnProb_pct
0,12346.0,77556.46,12,75.0
1,16754.0,67502.47,29,67.0
2,17850.0,56600.08,155,73.0
3,13093.0,54943.65,55,68.0
4,15749.0,44534.30,3,86.0
5,15098.0,39916.50,3,91.0
6,13902.0,34095.26,5,81.0
7,13802.0,26259.11,19,73.0
8,12482.0,23691.40,29,6.0
9,14063.0,22710.20,9,75.0


In [ ]:
# Save all results for dashboard
q1 = pd.read_sql_query("""
    SELECT t.Country,
           COUNT(DISTINCT t.[Customer ID]) AS TotalCustomers,
           SUM(c.Churned) AS ChurnedCustomers,
           ROUND(SUM(c.Churned)*100.0/COUNT(DISTINCT t.[Customer ID]),2) AS ChurnRate
    FROM transactions t
    JOIN customers c ON t.[Customer ID] = c.[Customer ID]
    GROUP BY t.Country
    HAVING TotalCustomers > 10
    ORDER BY ChurnRate DESC
    LIMIT 10
""", conn)

q2 = pd.read_sql_query("""
    SELECT RiskSegment, COUNT(*) AS CustomerCount,
           ROUND(AVG(AvgOrderValue),2) AS AvgOrderValue,
           ROUND(AVG(Monetary),2) AS AvgTotalSpend
    FROM customers
    GROUP BY RiskSegment
""", conn)

q4 = pd.read_sql_query("""
    SELECT SUBSTR(InvoiceDate,1,7) AS Month,
           ROUND(SUM(Quantity*Price),2) AS TotalRevenue
    FROM transactions
    GROUP BY Month
    ORDER BY Month
""", conn)

q1.to_csv('/content/drive/MyDrive/retail-churn-project/data/churn_by_country.csv', index=False)
q2.to_csv('/content/drive/MyDrive/retail-churn-project/data/risk_segments.csv', index=False)
q4.to_csv('/content/drive/MyDrive/retail-churn-project/data/monthly_revenue.csv', index=False)

print("✅ All query results saved!")



✅ All query results saved!
